<a href="https://colab.research.google.com/github/Ranjitkumarmuduli/tomato-disease-detection/blob/main/05_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 5 – Web Application (Tomato Leaf Doctor)
A Gradio web app: upload a tomato leaf photo → disease prediction + confidence →
Grad-CAM heatmap → treatment and prevention advice. Deployed on Hugging Face Spaces.


## 1. Install libraries
- `gradio`: builds the web interface with a few lines of Python
- `huggingface_hub`: uploads the app to Hugging Face Spaces (free hosting)

In [1]:
!pip -q install timm gradio huggingface_hub kagglehub

## 2. Prepare the app folder
The app folder on Drive (`tomato_project/app_model/`) becomes the website.
We add one **test** image per class as clickable examples for the demo.

In [3]:
import os, sys, shutil, pandas as pd, kagglehub
from google.colab import drive
drive.mount('/content/drive')

PROJ = "/content/drive/MyDrive/tomato_project"
APP  = f"{PROJ}/app_model"
base = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
ROOT = f"{base}/plantvillage dataset/color"

os.makedirs(f"{APP}/examples", exist_ok=True)
test_df = pd.read_csv(f"{PROJ}/splits/test.csv")
for _, row in test_df.groupby("label").head(1).iterrows():          # first test image of each class
    shutil.copy(f"{ROOT}/{row.relpath}", f"{APP}/examples/{row['class'].replace(' ', '_')}.jpg")
print(sorted(os.listdir(f"{APP}/examples")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
['Bacterial_spot.jpg', 'Early_blight.jpg', 'Late_blight.jpg', 'Leaf_Mold.jpg', 'Septoria_leaf_spot.jpg', 'Spider_mites_Two-spotted_spider_mite.jpg', 'Target_Spot.jpg', 'Tomato_Yellow_Leaf_Curl_Virus.jpg', 'Tomato_mosaic_virus.jpg', 'healthy.jpg']


## 3. The application code
`%%writefile` saves this cell as the file `app.py` in the app folder. Main parts:
- **Load:** model, class names, threshold (`config.json`) and advice (`disease_info.json`)
- **Leaf check:** share of plant-coloured pixels (a simple safety filter)
- **Grad-CAM:** heatmap of where the model looked (same method as Step 3)
- **Advice card / Not-sure card:** the text shown to the farmer
- **Interface:** image input, Diagnose button, prediction bars, heatmap, advice

In [4]:
%%writefile /content/drive/MyDrive/tomato_project/app_model/app.py
"""Tomato Leaf Doctor – tomato leaf disease detection and treatment advice (MSc project)."""
import os, json
import numpy as np
import torch, timm
import torch.nn.functional as F
import gradio as gr
import matplotlib
from PIL import Image
from torchvision import transforms as T

# ---------- 1. Load model, settings and advice ----------
HERE = os.path.dirname(os.path.abspath(__file__))
CFG = json.load(open(os.path.join(HERE, "config.json")))
KB = json.load(open(os.path.join(HERE, "disease_info.json"), encoding="utf-8"))
CLASSES, THRESHOLD = CFG["classes"], CFG["confidence_threshold"]

model = timm.create_model(CFG["model_name"], pretrained=False, num_classes=len(CLASSES))
model.load_state_dict(torch.load(os.path.join(HERE, "model.pt"), map_location="cpu"))
model.eval()

preprocess = T.Compose([T.Resize((CFG["img_size"], CFG["img_size"])), T.ToTensor(),
                        T.Normalize(CFG["mean"], CFG["std"])])
JET = matplotlib.colormaps["jet"]

def pretty(name):
    return KB["diseases"][name]["display_name"]

# ---------- 2. Leaf check: is there enough plant colour (green/yellow/brown)? ----------
def plant_fraction(img):
    hsv = np.asarray(img.resize((128, 128)).convert("HSV")).astype(int)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]          # PIL hue scale 0-255
    plant = (h >= 15) & (h <= 120) & (s > 40) & (v > 40)     # ~20°-170°: brown → yellow → green
    return plant.mean()

# ---------- 3. Prediction + Grad-CAM ----------
def predict_with_cam(img):
    x = preprocess(img).unsqueeze(0)
    feats = model.forward_features(x)          # last feature maps
    feats.retain_grad()
    logits = model.forward_head(feats)
    probs = torch.softmax(logits, dim=1)[0]
    cls = int(probs.argmax())
    model.zero_grad()
    logits[0, cls].backward()
    weights = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * feats).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return probs.detach().numpy(), cls, cam.detach().numpy()

def overlay(img, cam):
    base = np.asarray(img.resize((cam.shape[1], cam.shape[0])), dtype=np.float32) / 255
    heat = JET(cam)[..., :3]
    return (np.clip(0.55 * base + 0.45 * heat, 0, 1) * 255).astype(np.uint8)

# ---------- 4. Text shown to the farmer ----------
def bullets(items):
    return "\n".join(f"- {x}" for x in items) if items else "- Not needed"

def advice_card(name, conf):
    d = KB["diseases"][name]
    if name == "healthy":
        return f"""## Healthy leaf ({conf:.0%} confidence)
No disease detected. Keep your crop healthy:
{bullets(d['prevention'])}"""
    return f"""## {d['display_name']} ({conf:.0%} confidence)
**Type:** {d['type']} | **Cause:** {d['pathogen']} | **Urgency:** {d['urgency']}

### Symptoms
{bullets(d['symptoms'])}

**How it spreads:** {d['spread']}

### What to do now – organic / cultural control
{bullets(d['organic_control'])}

### Chemical control (active ingredients)
{bullets(d['chemical_control'])}

### Prevention for the next crop
{bullets(d['prevention'])}

---
*{KB['disclaimer']}*
*Sources: {'; '.join(KB['sources'])}*"""

def not_sure_card(probs):
    top = np.argsort(probs)[::-1][:3]
    lines = "\n".join(f"- {pretty(CLASSES[i])}: {probs[i]:.0%}" for i in top)
    return f"""## Not sure ({probs[top[0]]:.0%} confidence)
The model is not confident enough to give treatment advice. Most likely:
{lines}

**Try again with a clearer photo:** one leaf, good daylight, plain background, in focus.
If symptoms continue, consult your local agriculture officer or Krishi Vigyan Kendra."""

NOT_LEAF = """## This doesn't look like a leaf
Please upload a close-up photo of a single tomato leaf."""

# ---------- 5. Main function called by the button ----------
def diagnose(img):
    if img is None:
        return {}, None, "Please upload a photo of a tomato leaf."
    img = img.convert("RGB")
    if plant_fraction(img) < 0.05:
        return {}, None, NOT_LEAF
    probs, cls, cam = predict_with_cam(img)
    top3 = {pretty(CLASSES[i]): float(probs[i]) for i in np.argsort(probs)[::-1][:3]}
    heat = overlay(img, cam)
    if probs[cls] < THRESHOLD:
        return top3, heat, not_sure_card(probs)
    return top3, heat, advice_card(CLASSES[cls], float(probs[cls]))

# ---------- 6. Web interface ----------
EXAMPLES = os.path.join(HERE, "examples")
with gr.Blocks(title="Tomato Leaf Doctor", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Tomato Leaf Doctor\nUpload a photo of a tomato leaf to detect disease and get "
                "treatment advice. Detects 9 diseases and healthy leaves.")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="pil", label="Tomato leaf photo", sources=["upload", "webcam"])
            btn = gr.Button("Diagnose", variant="primary")
            if os.path.isdir(EXAMPLES):
                gr.Examples(sorted(os.path.join(EXAMPLES, f) for f in os.listdir(EXAMPLES)), inputs=inp)
        with gr.Column():
            label_out = gr.Label(num_top_classes=3, label="Prediction")
            cam_out = gr.Image(label="Where the model looked (Grad-CAM heatmap)")
    card_out = gr.Markdown()
    btn.click(diagnose, inputs=inp, outputs=[label_out, cam_out, card_out])
    gr.Markdown(f"<small>Model: {CFG['model_name']} fine-tuned on PlantVillage tomato images "
                "(test accuracy 99.7%). Trained on lab photos; accuracy on field photos may be lower. "
                "MSc project – advisory use only.</small>")

if __name__ == "__main__":
    demo.launch()

Writing /content/drive/MyDrive/tomato_project/app_model/app.py


## 4. requirements.txt
Tells Hugging Face which libraries to install. The extra index installs the **CPU** version
of PyTorch, which is much smaller (the free Space has no GPU).

In [5]:
%%writefile /content/drive/MyDrive/tomato_project/app_model/requirements.txt
--extra-index-url https://download.pytorch.org/whl/cpu
torch
torchvision
timm
matplotlib

Writing /content/drive/MyDrive/tomato_project/app_model/requirements.txt


## 5. Test the app
Starts the app and prints a temporary public link (`*.gradio.live`, valid for 72 hours).
Open it and test:
1. Click each example image → correct disease + advice card
2. Upload a non-leaf photo (e.g. a face or a car) → "doesn't look like a leaf"
3. Upload a real tomato leaf photo from your phone or the internet

To stop the app, click the stop button on this cell.

In [ ]:
import importlib
os.chdir(APP); sys.path.insert(0, APP)
import app
importlib.reload(app)              # picks up changes if you edit Cell 3 and re-run it
app.demo.launch(share=True, debug=True)

## 6. Prepare the deployment folder
Hugging Face now requires a paid plan for Gradio apps, so we deploy on **Streamlit Community Cloud**
(free, deploys straight from GitHub). This cell collects everything the website needs into one `app/` folder.

In [13]:
DEPLOY = "/content/deploy/app"
os.makedirs(DEPLOY, exist_ok=True)
for f in ["model.pt", "config.json", "disease_info.json"]:
    shutil.copy(f"{APP}/{f}", DEPLOY)
shutil.copytree(f"{APP}/examples", f"{DEPLOY}/examples", dirs_exist_ok=True)
print(os.listdir(DEPLOY))

['model.pt', 'examples', 'disease_info.json', 'config.json']


## 7. Streamlit app (`streamlit_app.py`)
Same logic as the Gradio version (leaf check → prediction + Grad-CAM → confidence check → advice),
written for Streamlit. `@st.cache_resource` loads the model only once, so every click is fast.
Urgency is colour-coded: red = High, orange = Medium.

In [14]:
%%writefile /content/deploy/app/streamlit_app.py
"""Tomato Leaf Doctor – tomato leaf disease detection and treatment advice (MSc project).
Run locally:  streamlit run app/streamlit_app.py
"""
import os, json
import numpy as np
import torch, timm
import torch.nn.functional as F
import matplotlib
import streamlit as st
from PIL import Image
from torchvision import transforms as T

HERE = os.path.dirname(os.path.abspath(__file__))
st.set_page_config(page_title="Tomato Leaf Doctor", layout="wide")

# ---------- 1. Load model, settings and advice (cached: loaded once, not on every click) ----------
@st.cache_resource
def load_everything():
    cfg = json.load(open(os.path.join(HERE, "config.json")))
    kb = json.load(open(os.path.join(HERE, "disease_info.json"), encoding="utf-8"))
    model = timm.create_model(cfg["model_name"], pretrained=False, num_classes=len(cfg["classes"]))
    model.load_state_dict(torch.load(os.path.join(HERE, "model.pt"), map_location="cpu"))
    model.eval()
    return cfg, kb, model

CFG, KB, model = load_everything()
CLASSES, THRESHOLD = CFG["classes"], CFG["confidence_threshold"]
preprocess = T.Compose([T.Resize((CFG["img_size"], CFG["img_size"])), T.ToTensor(),
                        T.Normalize(CFG["mean"], CFG["std"])])
JET = matplotlib.colormaps["jet"]

def pretty(name):
    return KB["diseases"][name]["display_name"]

# ---------- 2. Leaf check: enough plant colour (brown / yellow / green)? ----------
def plant_fraction(img):
    hsv = np.asarray(img.resize((128, 128)).convert("HSV")).astype(int)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]          # PIL hue scale 0-255
    return ((h >= 15) & (h <= 120) & (s > 40) & (v > 40)).mean()

# ---------- 3. Prediction + Grad-CAM heatmap ----------
def predict_with_cam(img):
    x = preprocess(img).unsqueeze(0)
    feats = model.forward_features(x)          # last feature maps
    feats.retain_grad()
    logits = model.forward_head(feats)
    probs = torch.softmax(logits, dim=1)[0]
    cls = int(probs.argmax())
    model.zero_grad()
    logits[0, cls].backward()
    weights = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * feats).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return probs.detach().numpy(), cls, cam.detach().numpy()

def overlay(img, cam):
    base = np.asarray(img.resize((cam.shape[1], cam.shape[0])), dtype=np.float32) / 255
    heat = JET(cam)[..., :3]
    return (np.clip(0.55 * base + 0.45 * heat, 0, 1) * 255).astype(np.uint8)

# ---------- 4. Advice shown to the farmer ----------
def bullets(items):
    return "\n".join(f"- {x}" for x in items) if items else "- Not needed"

def show_advice(name, conf):
    d = KB["diseases"][name]
    if name == "healthy":
        st.success(f"Healthy leaf ({conf:.0%} confidence). No disease detected.")
        st.markdown("**Keep your crop healthy:**\n" + bullets(d["prevention"]))
        return
    level = {"High": st.error, "Medium": st.warning}.get(d["urgency"], st.info)
    level(f"{d['display_name']} detected ({conf:.0%} confidence). Urgency: {d['urgency']}")
    st.markdown(f"**Type:** {d['type']} | **Cause:** {d['pathogen']}")
    st.markdown("#### Symptoms\n" + bullets(d["symptoms"]))
    st.markdown(f"**How it spreads:** {d['spread']}")
    c1, c2 = st.columns(2)
    c1.markdown("#### What to do now (organic / cultural)\n" + bullets(d["organic_control"]))
    c2.markdown("#### Chemical control (active ingredients)\n" + bullets(d["chemical_control"]))
    st.markdown("#### Prevention for the next crop\n" + bullets(d["prevention"]))
    st.caption(KB["disclaimer"])
    st.caption("Sources: " + "; ".join(KB["sources"]))

# ---------- 5. Page layout ----------
st.title("Tomato Leaf Doctor")
st.write("Upload or take a photo of a tomato leaf to detect disease and get treatment advice. "
         "Detects 9 diseases and healthy leaves.")

source = st.radio("Image source", ["Upload a photo", "Use camera", "Try an example"], horizontal=True)
img = None
if source == "Upload a photo":
    f = st.file_uploader("Tomato leaf photo", type=["jpg", "jpeg", "png"])
    img = Image.open(f) if f else None
elif source == "Use camera":
    f = st.camera_input("Take a photo of one leaf")
    img = Image.open(f) if f else None
else:
    ex_dir = os.path.join(HERE, "examples")
    files = sorted(os.listdir(ex_dir)) if os.path.isdir(ex_dir) else []
    if files:
        choice = st.selectbox("Example image", files)
        img = Image.open(os.path.join(ex_dir, choice))

if img is not None:
    img = img.convert("RGB")
    left, right = st.columns(2)
    left.image(img, caption="Your photo", width="stretch")
    if plant_fraction(img) < 0.05:
        st.error("This doesn't look like a leaf. Please upload a close-up photo of a single tomato leaf.")
    else:
        with st.spinner("Analysing the leaf..."):
            probs, cls, cam = predict_with_cam(img)
        right.image(overlay(img, cam), caption="Where the model looked (Grad-CAM)", width="stretch")
        st.subheader("Top predictions")
        for i in np.argsort(probs)[::-1][:3]:
            st.progress(float(probs[i]), text=f"{pretty(CLASSES[i])}: {probs[i]:.0%}")
        st.divider()
        if probs[cls] < THRESHOLD:
            st.warning(f"Not sure ({probs[cls]:.0%} confidence). The model is not confident enough to give "
                       "treatment advice. Try a clearer photo: one leaf, good daylight, plain background. "
                       "If symptoms continue, consult your local agriculture officer or Krishi Vigyan Kendra.")
        else:
            show_advice(CLASSES[cls], float(probs[cls]))

st.divider()
st.caption(f"Model: {CFG['model_name']} fine-tuned on PlantVillage tomato images (test accuracy 99.7%). "
           "Trained on lab photos; accuracy on field photos may be lower. MSc project – advisory use only.")

Writing /content/deploy/app/streamlit_app.py


## 8. requirements.txt
Libraries Streamlit Cloud installs. The first line gets the small **CPU-only** PyTorch (the free server has no GPU).

In [15]:
%%writefile /content/deploy/app/requirements.txt
--extra-index-url https://download.pytorch.org/whl/cpu
torch
torchvision
timm
matplotlib
numpy
pillow
streamlit

Writing /content/deploy/app/requirements.txt


## 9. Download the app folder as a zip

In [16]:
shutil.make_archive("/content/tomato_app", "zip", "/content/deploy")
from google.colab import files
files.download("/content/tomato_app.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>